# leankg — manufacturing (日本語) demo

日本語の保全記録（手書きサンプル）から知識グラフを作ります。LLM は **オフ**（`llm="none"`）なので
API キーは不要です。最後に日本語の Cypher クエリを 3 つ実行します（Neo4j がなければ NetworkX で代替）。

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # so leankg finds domains/ and leankg.toml
from leankg import Pipeline
from leankg.report import render_report

In [ ]:
pipe = Pipeline.from_domain("manufacturing", llm="none")
report = pipe.build(ROOT / "examples" / "manufacturing" / "maintenance_log_ja.txt",
                    out="json", out_path="out/manufacturing_ja")
print(render_report(report))

## 抽出された関係（ルールのみ）

In [ ]:
ents = pipe.graph.entities
for t in pipe.graph.facts():
    h, tl = ents[t.head], ents[t.tail]
    neg = "（否定）" if t.negated else ""
    print(f"{h.name_ja or h.name} -[{t.type}{neg}]-> {tl.name_ja or tl.name}  ({t.method}, {t.confidence:.2f})")

## クエリ 3 つ

In [ ]:
# Connect to Neo4j if it's running (docker compose up -d); otherwise use NetworkX.
from leankg.config import load_config

cfg = load_config()
writer = None
try:
    from leankg.export.neo4j_writer import Neo4jWriter
    pw = cfg.neo4j.password.get_secret_value() if cfg.neo4j.password else "leankg-dev-pw"
    writer = Neo4jWriter(cfg.neo4j.uri, cfg.neo4j.user, pw)
    writer.driver.verify_connectivity()
    writer.clear()  # demo database only!
    print(writer.write_graph(pipe.graph))
    writer.run("CALL db.awaitIndexes(60)")
except Exception as exc:  # no server / no [neo4j] extra
    writer = None
    print(f"Neo4j not available ({type(exc).__name__}); falling back to NetworkX")

from leankg.export import to_networkx
G = to_networkx(pipe.graph)

def cypher(query, **params):
    if writer is None:
        return None
    return [dict(r) for r in writer.run(query, **params)]

In [ ]:
# 1. 全文検索（CJK アナライザ）: 「焼き付き」
rows = cypher("CALL db.index.fulltext.queryNodes('entity_names', $q) YIELD node, score "
              "RETURN node.id AS id, node.name_ja AS name_ja, score", q="焼き付き")
if rows is None:
    rows = [{"id": n, "name_ja": d.get("name_ja")} for n, d in G.nodes(data=True)
            if "焼き付き" in str(d.get("name_ja", "")) + str(d.get("aliases_text", ""))]
rows

In [ ]:
# 2. グラフで「なぜなぜ分析」: CAUSED_BY を最大 5 段たどる
rows = cypher("MATCH p=(f:FailureMode)-[:CAUSED_BY*1..5]->(c) "
              "WHERE all(r IN relationships(p) WHERE r.negated = false) "
              "RETURN [n IN nodes(p) | coalesce(n.name_ja, n.name)] AS chain")
if rows is None:
    rows = [{"chain": [G.nodes[u].get("name_ja"), G.nodes[v].get("name_ja")]}
            for u, v, k, d in G.edges(keys=True, data=True) if k == "CAUSED_BY" and d.get("negated") is False]
rows

In [ ]:
# 3. TPS 概念と防止対象
rows = cypher("MATCH (t:TPSConcept)-[:PREVENTS]->(x) RETURN t.name_ja AS 概念, collect(coalesce(x.name_ja, x.name)) AS 防止")
if rows is None:
    rows = [{"概念": G.nodes[u].get("name_ja"), "防止": G.nodes[v].get("name_ja")}
            for u, v, k, d in G.edges(keys=True, data=True) if k == "PREVENTS"]
rows